# Hypothesis Testing: Waiter's Tips

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [2]:
import seaborn as sns

tips = sns.load_dataset("tips")

Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

---

In [ ]:
import numpy as np
import pandas as pd

## H1: smoking is associated with time of visit

smoker and time are both categorical, so we use chi-square + cramers V

In [ ]:
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association

alpha = 0.05

contingency_h1 = pd.crosstab(index=tips['smoker'], columns=tips['time'])

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_h1)
cramers_v = association(observed=contingency_h1, method='cramer', correction=True)

print(f"p-value: {p_value:.4f} -> {'significant' if p_value <= alpha else 'not significant'}")
print(f"Cramers V: {cramers_v:.2f}, dof: {dof}")

## H2: the bigger the group the higher the tip

size and tip are both numerical, so we use pearson correlation

In [ ]:
from scipy.stats import pearsonr

result = pearsonr(tips['size'], tips['tip'])

print(f"r = {result.statistic:.2f}")
print(f"p-value: {result.pvalue:.4f} -> {'significant' if result.pvalue <= alpha else 'not significant'}")

## H3: group size is different based on time of visit

size is numerical, time is categorical (2 groups: Lunch/Dinner), so we use a t-test

In [ ]:
from scipy.stats import ttest_ind

lunch = tips[tips['time'] == 'Lunch']['size']
dinner = tips[tips['time'] == 'Dinner']['size']

result = ttest_ind(lunch, dinner)

print(f"p-value: {result.pvalue:.4f} -> {'significant' if result.pvalue <= alpha else 'not significant'}")

## H4: males tip more than females

tip is numerical, sex is categorical (2 groups), so we use a t-test

In [ ]:
male_tips = tips[tips['sex'] == 'Male']['tip']
female_tips = tips[tips['sex'] == 'Female']['tip']

result = ttest_ind(male_tips, female_tips)

print(f"male mean tip: {male_tips.mean():.2f}, female mean tip: {female_tips.mean():.2f}")
print(f"p-value: {result.pvalue:.4f} -> {'significant' if result.pvalue <= alpha else 'not significant'}")

## Confounder: does size cause higher tip, or just a higher bill?

check: size -> tip, size -> total_bill, total_bill -> tip
if size correlates with bill, and bill correlates with tip, then bill is a confounder

In [ ]:
r_size_tip, _ = pearsonr(tips['size'], tips['tip'])
r_size_bill, _ = pearsonr(tips['size'], tips['total_bill'])
r_bill_tip, _ = pearsonr(tips['total_bill'], tips['tip'])

print(f"size vs tip:        r = {r_size_tip:.2f}")
print(f"size vs total_bill: r = {r_size_bill:.2f}")
print(f"total_bill vs tip:  r = {r_bill_tip:.2f}")

# control for total_bill: check if size still correlates with tip after removing bill's effect
tips['tip_residual'] = tips['tip'] - r_bill_tip * tips['total_bill']
r_partial, _ = pearsonr(tips['size'], tips['tip_residual'])
print(f"\nsize vs tip (controlling for bill): r = {r_partial:.2f}")